In [ ]:
# ───  ติดตั้ง Dependencies + set apikey ───────────────────────────────────────────
!pip install opencv-python-headless numpy pillow pandas tqdm beautifulsoup4 typhoon-ocr rich

**อย่าลืมใส่API keyก่อน**


`api_key = "sk-...." `



In [ ]:
import os
import sys
import re
import time
import random
import warnings
from pathlib import Path
from collections import defaultdict

# --- 🔑 ตั้งค่า API Key ---
api_key = "sk-...." ### 🔑 ใส่API Key ก่อน ###
os.environ["TYPHOON_OCR_API_KEY"] = api_key.strip()  # ใช้ .strip() เพื่อลบช่องว่างที่อาจเผลอก๊อปติดมา

# นำเข้าไลบรารีสำหรับการทำงานแบบขนาน (Parallel & Concurrent)
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from threading import Semaphore

# นำเข้าไลบรารีจัดการข้อมูลและ HTML
import pandas as pd
from bs4 import BeautifulSoup
from typhoon_ocr import ocr_document  # ให้แน่ใจว่าไลบรารีนี้มีอยู่ใน Colab

# นำเข้าไลบรารีจัดการรูปภาพ
import cv2
import numpy as np
from PIL import Image

# 🌟 นำเข้าไลบรารี Rich สำหรับทำ UI ให้สวยงาม
from rich.console import Console
from rich.progress import Progress, SpinnerColumn, TextColumn, BarColumn, TaskProgressColumn, TimeElapsedColumn
from rich.panel import Panel
from rich.theme import Theme
from rich import print as rprint

# สร้าง Theme กำหนดสีให้ดูง่าย
custom_theme = Theme({
    "info": "cyan",
    "warning": "bold yellow",
    "error": "bold red",
    "success": "bold green"
})
console = Console(theme=custom_theme)

# ปิดการแจ้งเตือนที่ไม่จำเป็นเพื่อให้หน้าจอสะอาด
warnings.filterwarnings("ignore")

# =============================================================================
# ⚙️ CONFIG & GLOBAL SETTINGS (การตั้งค่าส่วนกลาง)
# =============================================================================

DOC_WORKERS = 4          # จำนวน Thread สำหรับประมวลผลเอกสาร
PAGE_WORKERS = 2         # จำนวน Thread สำหรับประมวลผลหน้าย่อย
API_CONCURRENCY = 4      # จำนวนการเรียก API พร้อมกันสูงสุด
api_limit = Semaphore(API_CONCURRENCY)

RAW_RESULTS_DIR = "ocr_raw_results"
# ตารางแปลงตัวเลขไทยเป็นเลขอารบิก
THAI_DIGIT = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")


# =============================================================================
# 🎨 ส่วนที่ 1: IMAGE PREPROCESSING (การเตรียมและปรับแต่งรูปภาพ)
# =============================================================================

def is_incomplete_partylist(html_content, expected=57):
    """ตรวจสอบว่าข้อมูลปาร์ตี้ลิสต์มาครบหรือไม่ (เทียบกับจำนวนที่คาดหวัง)"""
    votes = extract_partylist_votes(html_content)
    return len(votes) < expected * 0.9

def has_too_many_zeros(results, ids):
    """ตรวจสอบว่าผลลัพธ์มีค่า 0 มากเกินไปหรือไม่ (อาจเกิดจาก OCR พลาด)"""
    vals = [results.get(i, 0) for i in ids]
    return vals.count(0) / len(vals) > 0.1

def laplacian_variance(gray: np.ndarray) -> float:
    """คำนวณหาความเบลอของภาพ (ค่าความแปรปรวน Laplacian)"""
    return cv2.Laplacian(gray, cv2.CV_64F).var()

def is_blank_page(gray: np.ndarray, threshold: float = 250.0) -> bool:
    """ตรวจสอบว่าเป็นกระดาษเปล่าหรือไม่ โดยดูจากค่าเฉลี่ยสี"""
    return gray.mean() > threshold

def has_table(gray: np.ndarray, min_lines: int = 4, min_cells: int = 6, min_area_ratio: float = 0.05) -> bool:
    """ตรวจสอบว่าภาพนี้มีโครงสร้างของ 'ตาราง' อยู่หรือไม่"""
    H, W = gray.shape
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # ค้นหาเส้นแนวนอน
    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
    h_lines  = cv2.morphologyEx(binary, cv2.MORPH_OPEN, h_kernel)
    h_count  = int((cv2.reduce(h_lines, 1, cv2.REDUCE_MAX) > 0).sum())

    # ค้นหาเส้นแนวตั้ง
    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 40))
    v_lines  = cv2.morphologyEx(binary, cv2.MORPH_OPEN, v_kernel)
    v_count  = int((cv2.reduce(v_lines, 0, cv2.REDUCE_MAX) > 0).sum())

    if h_count < min_lines or v_count < 2:
        return False

    # รวมเส้นเพื่อหาพื้นที่ตาราง
    table_mask = cv2.add(h_lines, v_lines)
    contours, _ = cv2.findContours(table_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return False

    # ตรวจสอบสัดส่วนพื้นที่ตารางเทียบกับภาพรวม
    x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
    area_ratio = (w * h) / (W * H)
    if area_ratio < min_area_ratio:
        return False

    return True

def crop_to_table(gray: np.ndarray, padding: int = 30) -> np.ndarray:
    """ตัดภาพ (Crop) ให้เหลือเฉพาะบริเวณที่เป็นตาราง"""
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (50, 1))
    h_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, h_kernel)

    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 50))
    v_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, v_kernel)

    table_mask = cv2.add(h_lines, v_lines)
    contours, _ = cv2.findContours(table_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return gray

    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)

    H, W = gray.shape
    x1, y1 = max(0, x - padding), max(0, y - padding)
    x2, y2 = min(W, x + w + padding), min(H, y + h + padding)

    return gray[y1:y2, x1:x2]

def deskew(gray: np.ndarray) -> np.ndarray:
    """ปรับแก้ความเอียงของรูปภาพ (Deskewing)"""
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLines(edges, 1, np.pi / 180, threshold=200)

    if lines is None:
        return gray

    angles = [np.degrees(theta) - 90 for rho, theta in lines[:, 0] if abs(np.degrees(theta) - 90) < 10]
    if not angles:
        return gray

    median_angle = float(np.median(angles))
    if abs(median_angle) < 0.3:
        return gray

    h, w = gray.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), median_angle, 1.0)
    return cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)

def enhance_clarity(gray: np.ndarray, blur_score: float) -> np.ndarray:
    """ปรับความคมชัดของภาพ และลด Noise"""
    denoised = cv2.fastNlMeansDenoising(gray, h=8, templateWindowSize=7, searchWindowSize=21)

    # ปรับ Contrast ด้วย CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    contrasted = clahe.apply(denoised)

    # ปรับความแรงของการทำ Sharpening ตามค่าความเบลอ
    strength = 1.8 if blur_score < 50 else 1.2 if blur_score < 100 else 0.8
    blurred = cv2.GaussianBlur(contrasted, (0, 0), 3)
    sharpened = cv2.addWeighted(contrasted, 1 + strength, blurred, -strength, 0)

    return cv2.normalize(sharpened, None, 0, 255, cv2.NORM_MINMAX)

def maybe_upscale(img: np.ndarray, min_height: int = 1500) -> np.ndarray:
    """ขยายขนาดภาพหากภาพมีความสูงน้อยเกินไป เพื่อให้ OCR อ่านง่ายขึ้น"""
    h, w = img.shape[:2]
    if h < min_height:
        scale = min_height / h
        img = cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_LANCZOS4)
    return img

def detect_grid_lines(gray: np.ndarray) -> tuple[list, list]:
    """ตรวจจับและคืนค่าตำแหน่งเส้นตาราง (แนวนอนและแนวตั้ง)"""
    h, w = gray.shape[:2]
    if h < 40 or w < 40:
        return [], []

    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (max(30, w // 25), 1))
    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(30, h // 25)))

    h_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, h_kernel)
    v_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, v_kernel)

    h_lines = cv2.dilate(h_lines, cv2.getStructuringElement(cv2.MORPH_RECT, (15, 1)), iterations=1)
    v_lines = cv2.dilate(v_lines, cv2.getStructuringElement(cv2.MORPH_RECT, (1, 15)), iterations=1)

    h_segments, v_segments = [], []

    # หาเส้นแนวนอน
    h_contours, _ = cv2.findContours(h_lines, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in h_contours:
        x, y, seg_w, seg_h = cv2.boundingRect(cnt)
        if seg_w < int(w * 0.18):
            continue
        h_segments.append((x, y + seg_h // 2, x + seg_w, y + seg_h // 2))

    # หาเส้นแนวตั้ง
    v_contours, _ = cv2.findContours(v_lines, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in v_contours:
        x, y, seg_w, seg_h = cv2.boundingRect(cnt)
        if seg_h < int(h * 0.18):
            continue
        v_segments.append((x + seg_w // 2, y, x + seg_w // 2, y + seg_h))

    h_segments.sort(key=lambda p: p[1])
    v_segments.sort(key=lambda p: p[0])

    return h_segments, v_segments

def draw_grid_enhancement(gray: np.ndarray) -> tuple[np.ndarray, int, int]:
    """วาดเส้นทับโครงสร้างตารางเดิมเพื่อให้ชัดเจนขึ้น (สีเขียว-แนวนอน, สีแดง-แนวตั้ง)"""
    h_segments, v_segments = detect_grid_lines(gray)
    color_img = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    thickness = max(2, min(gray.shape[:2]) // 450)

    for x1, y1, x2, y2 in h_segments:
        cv2.line(color_img, (x1, y1), (x2, y2), (0, 255, 0), thickness=thickness, lineType=cv2.LINE_AA)
    for x1, y1, x2, y2 in v_segments:
        cv2.line(color_img, (x1, y1), (x2, y2), (0, 0, 255), thickness=thickness, lineType=cv2.LINE_AA)

    return color_img, len(h_segments), len(v_segments)

def preprocess_image(src_path: Path, dst_path: Path) -> dict:
    """ท่อประมวลผลหลักในการเตรียมภาพก่อนส่งเข้า OCR"""
    img_bgr = cv2.imread(str(src_path))
    if img_bgr is None:
        raise ValueError(f"ไม่สามารถอ่านรูปได้: {src_path}")

    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # ข้ามรูปที่ไม่จำเป็น
    if is_blank_page(gray): return {"file": src_path.name, "skipped": True}
    if not has_table(gray): return {"file": src_path.name, "skipped": True}

    # กระบวนการปรับแต่ง
    gray = maybe_upscale(gray)
    blur = laplacian_variance(gray)
    gray = deskew(gray)
    gray = crop_to_table(gray)
    enhanced = enhance_clarity(gray, blur)
    enhanced = maybe_upscale(enhanced, min_height=2000)
    enhanced_color, _, _ = draw_grid_enhancement(enhanced)

    # บันทึกไฟล์
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(dst_path), enhanced_color, [cv2.IMWRITE_PNG_COMPRESSION, 1])
    return {"file": src_path.name, "skipped": False}

def _process_single_task(args):
    """ฟังก์ชันย่อยสำหรับการทำ Threading (แต่งภาพเดี่ยว)"""
    path, dst = args
    out_path = dst / path.name

    if out_path.exists():
        return {"file": path.name, "cached": True}

    try:
        return preprocess_image(path, out_path)
    except Exception as e:
        return {"file": path.name, "error": str(e), "skipped": True}

def run_batch_preprocess(input_dir: str, output_dir: str):
    """รันการแต่งภาพทั้งหมดในโฟลเดอร์แบบขนาน (Parallel)"""
    src, dst = Path(input_dir), Path(output_dir)
    dst.mkdir(parents=True, exist_ok=True)

    images = []
    for ext in ["*.jpg", "*.jpeg", "*.png"]:
        images.extend(list(src.glob(ext)))

    if not images:
        return []

    console.print(f"\n[bold cyan]🔧 กำลังเตรียมภาพ {len(images)} รูป → {dst} (ใช้งาน Thread ทุก Core)[/bold cyan]")
    tasks = [(path, dst) for path in images]

    results = []
    # 🌟 เปลี่ยนมาใช้ Progress Bar ของ Rich
    with Progress(
        SpinnerColumn(),
        TextColumn("[progress.description]{task.description}"),
        BarColumn(),
        TaskProgressColumn(),
        TimeElapsedColumn(),
        console=console
    ) as progress:
        task_id = progress.add_task("[blue]กำลังแต่งภาพให้ชัดเจน...", total=len(tasks))

        with ThreadPoolExecutor() as executor:
            # ใช้ as_completed เพื่ออัปเดตหลอดโหลดตามงานที่เสร็จจริง
            futures = [executor.submit(_process_single_task, task) for task in tasks]
            for future in as_completed(futures):
                results.append(future.result())
                progress.update(task_id, advance=1)

    return results

def patch_ocr_pipeline(images_dir: str) -> str:
    """ห่อหุ้มกระบวนการแต่งภาพ และคืนค่า Path โฟลเดอร์รูปที่แต่งเสร็จแล้ว"""
    clean_dir = str(Path(images_dir).parent / (Path(images_dir).name + "_clean"))
    run_batch_preprocess(images_dir, clean_dir)
    return clean_dir


# =============================================================================
# 🔍 ส่วนที่ 2: OCR & DATA PARSING (หมวดแกะตัวเลขและตรวจสอบความถูกต้อง)
# =============================================================================

def thai_text_to_num(text):
    """แปลงข้อความตัวอักษรไทยที่ระบุตัวเลข ให้เป็นจำนวนเต็ม (เช่น 'หนึ่งร้อย' -> 100)"""
    if not text: return 0
    text = text.replace('(', '').replace(')', '').strip()

    digits = {'ศูนย์':0, 'หนึ่ง':1, 'เอ็ด':1, 'สอง':2, 'ยี่':2, 'สาม':3, 'สี่':4, 'ห้า':5, 'หก':6, 'เจ็ด':7, 'แปด':8, 'เก้า':9}
    positions = {'สิบ':10, 'ร้อย':100, 'พัน':1000, 'หมื่น':10000, 'แสน':100000, 'ล้าน':1000000}

    total, current_digit, i = 0, 0, 0

    while i < len(text):
        matched_pos = False
        # ตรวจสอบหลัก (สิบ, ร้อย, พัน, ...)
        for p_word, p_val in positions.items():
            if text[i:].startswith(p_word):
                if current_digit == 0 and p_word in ['สิบ', 'ร้อย', 'พัน', 'หมื่น', 'แสน', 'ล้าน']:
                     if p_word == 'สิบ': current_digit = 1
                total += current_digit * p_val
                current_digit = 0
                i += len(p_word)
                matched_pos = True
                break

        if matched_pos:
            continue

        # ตรวจสอบตัวเลข (หนึ่ง, สอง, สาม, ...)
        for d_word, d_val in digits.items():
            if text[i:].startswith(d_word):
                current_digit = d_val
                i += len(d_word)
                break
        else:
            i += 1

    total += current_digit
    return total

def extract_votes_list(html_content):
    """ดึงข้อมูลคะแนนเสียงจากตาราง HTML (สำหรับ ส.ส. เขต)"""
    if isinstance(html_content, list):
        html_content = "".join(str(x) for x in html_content)
    elif not isinstance(html_content, str):
        return []

    html_content = html_content.translate(THAI_DIGIT)
    soup = BeautifulSoup(html_content, 'html.parser')
    votes_list = []

    for row in soup.find_all('tr'):
        cols = row.find_all(['td', 'th'])
        if len(cols) >= 2:
            vote_text = cols[-1].get_text(strip=True)
            num_match = re.search(r'^[\d,\.]+', vote_text)

            # ถ้าหาในคอลัมน์สุดท้ายไม่เจอ ให้ลองหาในคอลัมน์รองสุดท้าย
            if not num_match and len(cols) >= 3:
                vote_text = cols[-2].get_text(strip=True)
                num_match = re.search(r'^[\d,\.]+', vote_text)

            text_match = re.search(r'\((.*?)\)', vote_text)
            if num_match:
                clean_num_str = num_match.group().replace(',', '').replace('.', '')
                if clean_num_str.isdigit():
                    num_val = int(clean_num_str)

                    # ตรวจสอบเทียบกับคำอ่านภาษาไทยในวงเล็บ (ถ้ามี)
                    if text_match:
                        thai_val = thai_text_to_num(text_match.group(1))
                        if thai_val > 0 and abs(num_val - thai_val) > 10:
                            votes_list.extend([num_val, thai_val])
                        else:
                            votes_list.append(num_val)
                    else:
                        votes_list.append(num_val)

    return votes_list

def extract_partylist_votes(html_content):
    """ดึงข้อมูลคะแนนเสียงแบบบัญชีรายชื่อ (Party List) โดยจับคู่เบอร์พรรคกับคะแนน"""
    if isinstance(html_content, list):
        html_content = "".join(str(x) for x in html_content)

    html_content = str(html_content).translate(THAI_DIGIT)
    soup = BeautifulSoup(html_content, 'html.parser')
    party_votes = {}

    for row in soup.find_all('tr'):
        cols = row.find_all(['td', 'th'])
        if len(cols) >= 2:
            # หาเบอร์พรรคจากคอลัมน์แรก
            match_party = re.search(r'^(\d+)', cols[0].get_text(strip=True))
            if match_party:
                party_num = int(match_party.group(1))

                # หาคะแนนจากคอลัมน์สุดท้าย
                vote_text = cols[-1].get_text(strip=True)
                num_match = re.search(r'^[\d,\.]+', vote_text)

                if not num_match and len(cols) >= 3:
                    vote_text = cols[-2].get_text(strip=True)
                    num_match = re.search(r'^[\d,\.]+', vote_text)

                if num_match:
                    clean_str = num_match.group().replace(',', '').replace('.', '')
                    if clean_str.isdigit():
                        party_votes[party_num] = int(clean_str)

    return party_votes

def validate_partylist_sequence(html_content):
    """ตรวจสอบว่าหมายเลขพรรคเรียงลำดับถูกต้องหรือไม่ (1, 2, 3...)"""
    if isinstance(html_content, list):
        html_content = "".join(str(x) for x in html_content)

    soup = BeautifulSoup(str(html_content).translate(THAI_DIGIT), 'html.parser')
    party_numbers = []

    for row in soup.find_all('tr'):
        cols = row.find_all(['td', 'th'])
        if len(cols) >= 2:
            match = re.search(r'^(\d+)', cols[0].get_text(strip=True))
            if match:
                party_numbers.append(int(match.group(1)))

    # ตรวจสอบความต่อเนื่อง
    if len(party_numbers) > 1:
        for i in range(len(party_numbers) - 1):
            if party_numbers[i] + 1 != party_numbers[i+1]:
                return False, party_numbers
    return True, party_numbers

def validate_total_sum(html_content):
    """ตรวจสอบผลรวมคะแนนเสียง ว่าคะแนนรายคนรวมกันเท่ากับช่องรวมคะแนนหรือไม่"""
    votes = extract_votes_list(html_content)
    if len(votes) >= 2:
        calc_sum = sum(votes[:-1])
        total_val = votes[-1]
        return (calc_sum == total_val), calc_sum, total_val
    return True, 0, 0

def process_doc(image_paths, _ui_progress=None):
    """ส่งรูปภาพเข้า API OCR (รองรับการทำงานแบบ Threading และการส่งซ้ำอัตโนมัติเมื่อ Timeout)"""
    def _process_single_image(img):
        max_retries = 6
        for attempt in range(max_retries):
            try:
                # จำกัดการเรียก API พร้อมกันด้วย Semaphore
                with api_limit:
                    res = ocr_document(img)
                    return str(res)
            except Exception as e:
                msg = str(e).lower()
                # 🛠️ แก้ไข: เพิ่มเงื่อนไข 408 และ timeout ให้ระบบทำการส่งซ้ำอัตโนมัติ (Exponential Backoff)
                if "429" in msg or "rate limit" in msg or "408" in msg or "timeout" in msg:
                    if attempt < max_retries - 1:
                        wait = (2 ** (attempt + 1)) + random.uniform(0, 1)
                        # ใช้ UI Progress ในการแสดงผลแจ้งเตือนถ้ามี
                        if _ui_progress:
                            _ui_progress.console.print(f"[warning]⏳ เซิร์ฟเวอร์ตอบช้า พัก {wait:.1f} วิ... (ส่ง {Path(img).name} ใหม่รอบ {attempt+1})[/warning]")
                        time.sleep(wait)
                    else:
                        raise Exception(f"🔥 ล้มเหลวถาวรหลังจากพยายามหลายครั้ง: {e}")
                else:
                    raise e
        return ""

    with ThreadPoolExecutor(max_workers=PAGE_WORKERS) as executor:
        html_results = list(executor.map(_process_single_image, image_paths))
    return "".join(html_results)

def process_with_fallback(doc, clean_paths, orig_paths, _ui_progress=None):
    """ประมวลผล OCR หากข้อมูลที่แต่งแล้วมีปัญหา จะลองใช้ภาพต้นฉบับแทน (Fallback)"""
    html_result = process_doc(clean_paths, _ui_progress)
    needs_fallback = False

    # 1. ตรวจสอบจากผลรวม
    is_sum_valid, _, _ = validate_total_sum(html_result)
    if not is_sum_valid:
        needs_fallback = True

    # 2. ตรวจสอบตามประเภทเอกสาร
    if "party_list" in doc.lower():
        is_seq_valid, _ = validate_partylist_sequence(html_result)
        if not is_seq_valid:
            needs_fallback = True

    elif "constituency" in doc.lower():
        votes = extract_votes_list(html_result)
        if len(votes) > 2:
            cand_votes = votes[:-1]
            if not all(cand_votes[i] >= cand_votes[i+1] for i in range(len(cand_votes)-1)):
                needs_fallback = True

    # ทำการ Fallback ถ้าจำเป็นและมีภาพต้นฉบับ
    if needs_fallback and clean_paths != orig_paths:
        return process_doc(orig_paths, _ui_progress), True

    return html_result, False

def process_doc_with_cache(doc, clean_paths, orig_paths, _ui_progress=None):
    """เรียกใช้ OCR โดยตรวจสอบ Cache ก่อน เพื่อประหยัด API ถ้าเคยรันไปแล้ว"""
    cache_path = os.path.join(RAW_RESULTS_DIR, f"{doc}.html")
    if os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8") as f:
            return f.read(), False, True

    html, fallback = process_with_fallback(doc, clean_paths, orig_paths, _ui_progress)
    return html, fallback, False


# =============================================================================
# 🚀 ส่วนที่ 3: MAIN EXECUTION (ส่วนสั่งการหลัก)
# =============================================================================

# จำลองการส่งค่า Arguments ผ่านคลาส แทนการใช้ argparse
class ColabConfig:
    def __init__(self):
        self.images_dir = "images"  # ชื่อโฟลเดอร์ภาพที่อัปโหลด
        self.template = "submission_template_v4.csv" # ชื่อไฟล์เทมเพลต CSV
        self.output = "submission.csv" # ชื่อไฟล์ผลลัพธ์ที่ต้องการ

if __name__ == "__main__":

    # --- 1. ตั้งค่าเริ่มต้น ---
    args = ColabConfig()

    # 🌟 สร้างกรอบข้อความแนะนำตัวสวยๆ
    welcome_panel = Panel(
        f"[info]📂 โฟลเดอร์รูป:[/info] [bold white]{args.images_dir}[/bold white]\n"
        f"[info]📄 ไฟล์แม่แบบ:[/info] [bold white]{args.template}[/bold white]\n"
        f"[info]💾 ปลายทาง:[/info] [bold white]{args.output}[/bold white]",
        title="[bold green]🚀 เริ่มกระบวนการแกะข้อมูลเลือกตั้ง[/bold green]",
        expand=False
    )
    console.print(welcome_panel)

    os.makedirs(RAW_RESULTS_DIR, exist_ok=True)

    # --- 2. ทำความสะอาดรูปภาพแบบรวดเร็ว ---
    clean_images_dir = patch_ocr_pipeline(args.images_dir)

    # --- 3. จัดกลุ่มเอกสารและรูปภาพ (อ้างอิงจากไฟล์ CSV) ---
    df = pd.read_csv(args.template)
    doc_index = defaultdict(list)

    for idx in df["id"]:
        parts = idx.split('_')
        doc_name = "_".join(parts[:-1])
        doc_index[doc_name].append(idx)

    doc_img_clean, doc_img_orig = defaultdict(list), defaultdict(list)

    for ext in ["*.jpg", "*.jpeg", "*.png"]:
        for orig_path in Path(args.images_dir).rglob(ext):
            name = orig_path.stem
            doc_name = re.sub(r'_page\d+$', '', name)

            doc_img_orig[doc_name].append(str(orig_path))

            clean_path = Path(clean_images_dir) / f"{name}.png"
            if not clean_path.exists():
                clean_path = Path(clean_images_dir) / orig_path.name

            doc_img_clean[doc_name].append(str(clean_path) if clean_path.exists() else str(orig_path))

    # เรียงลำดับหน้าเอกสาร
    for d in doc_img_orig: doc_img_orig[d].sort()
    for d in doc_img_clean: doc_img_clean[d].sort()

    docs_to_run = [d for d in doc_index if d in doc_img_clean]

    console.print(f"\n[bold magenta]🔥 กำลังประมวลผล OCR ทั้งหมด {len(docs_to_run)} ชุดเอกสาร[/bold magenta]")
    results = {}

    # --- 4. เริ่มต้นกระบวนการ OCR (ประมวลผลแบบขนาน) ---
    # 🌟 ใช้ Progress Bar ของ Rich เพื่อให้แจ้งเตือนต่างๆ ไม่ไปตีกับหลอดโหลด
    with Progress(
        SpinnerColumn(),
        TextColumn("[progress.description]{task.description}"),
        BarColumn(bar_width=40),
        TaskProgressColumn(),
        TimeElapsedColumn(),
        console=console
    ) as progress:

        ocr_task = progress.add_task("[bold magenta]แกะตัวเลขคะแนน...", total=len(docs_to_run))

        with ThreadPoolExecutor(max_workers=DOC_WORKERS) as executor:
            # ส่ง object progress เข้าไปด้วยเพื่อให้ข้างในพิมพ์ Warning ได้ถูกต้อง
            futures = { executor.submit(process_doc_with_cache, d, doc_img_clean[d], doc_img_orig[d], progress): d for d in docs_to_run }

            for f in as_completed(futures):
                doc = futures[f]
                try:
                    raw_html, used_fallback, used_cache = f.result()

                    if used_fallback:
                        progress.console.print(f"[warning]🔄 กู้คืนข้อมูล (Fallback):[/warning] '{doc}' ดึงภาพต้นฉบับมาซ่อมให้แล้ว!")

                    if not used_cache:
                        with open(os.path.join(RAW_RESULTS_DIR, f"{doc}.html"), "w", encoding="utf-8") as cache_file:
                            cache_file.write(raw_html)

                    # --- ตรวจสอบคะแนนขั้นตอนสุดท้าย ---
                    is_sum_ok, calc, actual = validate_total_sum(raw_html)
                    if not is_sum_ok and actual != 0:
                        progress.console.print(f"[warning]⚠️ ผลรวมไม่ตรง:[/warning] '{doc}' (คำนวณได้ [bold white]{calc}[/bold white] แต่ใบสรุปคือ [bold white]{actual}[/bold white])")

                    # แยกลอจิกการดึงคะแนนตามประเภทใบเลือกตั้ง
                    if "party_list" in doc.lower():
                        is_seq_ok, nums = validate_partylist_sequence(raw_html)
                        if not is_seq_ok:
                            progress.console.print(f"[warning]⚠️ เลขพรรคหาย/ข้าม:[/warning] '{doc}' [เห็นเบอร์: {nums}]")

                        votes_dict = extract_partylist_votes(raw_html)
                        for id_val in doc_index[doc]:
                            party_no = int(id_val.split('_')[-1])
                            if party_no in votes_dict:
                                results[id_val] = votes_dict[party_no]

                    elif "constituency" in doc.lower():
                        votes = extract_votes_list(raw_html)
                        if len(votes) > 1:
                            cand_votes, total_vote = votes[:-1], votes[-1]

                            # ตรวจสอบว่าคะแนนเรียงลำดับลงมาถูกต้องหรือไม่
                            if not all(cand_votes[i] >= cand_votes[i+1] for i in range(len(cand_votes)-1)):
                                progress.console.print(f"[info]💡 เรียงลำดับใหม่:[/info] '{doc}' คะแนนสลับบรรทัด ระบบบังคับเรียงให้ใหม่แล้ว!")
                                cand_votes = sorted(cand_votes, reverse=True)

                            votes = cand_votes + [total_vote]

                        for id_val, v in zip(doc_index[doc], votes):
                            results[id_val] = v

                except Exception as e:
                    progress.console.print(f"[error]❌ เกิดข้อผิดพลาดในชุด {doc}:[/error] {e}")

                # อัปเดตหลอดโหลด
                progress.update(ocr_task, advance=1)

    # --- 5. สร้างไฟล์ CSV ---
    final_output = [{"id": i, "votes": results.get(i, 1)} for i in df["id"]]
    try:
        pd.DataFrame(final_output).to_csv(args.output, index=False)

        # 🌟 สรุปผลตอนจบด้วย Panel สวยๆ
        success_panel = Panel(
            f"✅ บันทึกไฟล์สำเร็จไปที่ → [bold white]{args.output}[/bold white]\n"
            f"📁 ตรวจสอบข้อมูลดิบได้ที่โฟลเดอร์: [bold white]{RAW_RESULTS_DIR}/[/bold white]",
            title="[bold green]🎉 กระบวนการเสร็จสมบูรณ์![/bold green]",
            expand=False,
            border_style="green"
        )
        console.print(success_panel)

    except Exception as e:
        alt_name = args.output.replace(".csv", "_backup.csv")
        pd.DataFrame(final_output).to_csv(alt_name, index=False)

        error_panel = Panel(
            f"บันทึกทับไม่ได้ (อาจเปิดไฟล์เก่าค้างไว้ในโปรแกรมอื่น)\n"
            f"เซฟไปที่ชื่อสำรอง → [bold yellow]{alt_name}[/bold yellow]",
            title="[bold red]⚠️ มีปัญหาในการบันทึกไฟล์[/bold red]",
            expand=False,
            border_style="red"
        )
        console.print(error_panel)